# CL-MEE: corrected, reproducible experiments

Runs the controlled simulation, sensitivity analysis and Foursquare NYC evaluation. The *saved historical outputs are not authoritative*; execute all cells sequentially from a clean kernel and read `paper_results.json`.

**Input:** place `dataset_TSMC2014_NYC.txt` beside this notebook. The full package includes the input data; the lightweight Overleaf ZIP does not.

Key fixes: deterministic time/poplularity/score tie resolution; no fictitious displayed positive reasons at D=0; hard display truncation distinct from the soft penalty; P3 as explicit ranking-identity control; structural assertions instead of old hardcoded numerical expectations.


In [93]:

from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.neighbors import BallTree

# -------------------------
# Reproducibility constants
# -------------------------
SIM_SEEDS = list(range(10))
SIM_USERS = 800
SIM_POIS = 600
SIM_CATEGORIES = 10
SIM_SESSIONS_PER_USER = 3
SIM_M = 40
SIM_RHO = 0.80
SIM_ETA = 0.008
SIM_CONTEXT_PROBS = (0.35, 0.40, 0.25)  # context 0/1/2
SIM_BUDGET_POLICY = (3, 2, 1)
SIM_LAMBDAS = (0.0, 0.035, 0.075)
SIM_EPSILONS = (0.0, 0.05, 0.10)
SIM_LATENT_WEIGHTS = np.array([0.50, 0.35, 0.15], dtype=float)
SIM_MODEL_WEIGHTS = np.array([0.52, 0.31, 0.17], dtype=float)
SIM_LATENT_NOISE = 0.205
SIM_SCORE_NOISE = 0.015
SIM_DIRICHLET_ALPHA = 0.40
SIM_DISTANCE_TAU = 0.25

# Foursquare file expected by the original experiment.
# Change this path only if your uploaded file has a different filename.
DATASET_PATH = Path("dataset_TSMC2014_NYC.txt")

# Generated outputs are recomputed on a fresh, sequential run.
# Score contribution coverage excludes unmodelled additive score noise.
import hashlib, platform, scipy, sklearn
print("Environment:", {"python": platform.python_version(), "numpy":np.__version__, "pandas":pd.__version__, "scipy":scipy.__version__, "sklearn":sklearn.__version__})
if DATASET_PATH.exists():
    print("Dataset SHA256:", hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest())

print("Simulation sessions:", len(SIM_SEEDS) * SIM_USERS * SIM_SESSIONS_PER_USER)
print("Expected Foursquare dataset:", DATASET_PATH)


Environment: {'python': '3.14.7', 'numpy': '2.5.3', 'pandas': '3.0.6', 'scipy': '1.18.1', 'sklearn': '1.9.1'}
Dataset SHA256: 2f39ce698e5bff6683c74bec30c436cc022c160b0f3d73c5268b55491b2445f9
Simulation sessions: 24000
Expected Foursquare dataset: dataset_TSMC2014_NYC.txt


## 1. Controlled simulation (Table 2)

In [94]:

BASE_WEIGHTS = SIM_MODEL_WEIGHTS
LATENT_WEIGHTS = SIM_LATENT_WEIGHTS


def generate_seed_cache(seed, latent_sigma=SIM_LATENT_NOISE, score_sigma=SIM_SCORE_NOISE,
                        alpha=SIM_DIRICHLET_ALPHA, tau=SIM_DISTANCE_TAU, M=SIM_M):
    """Generate one reproducible synthetic seed (800 users x 3 sessions)."""
    rng = np.random.default_rng(seed)
    U, P, C, S = SIM_USERS, SIM_POIS, SIM_CATEGORIES, SIM_SESSIONS_PER_USER

    poi_cat = rng.integers(C, size=P)
    poi_xy = rng.random((P, 2))
    popularity = rng.beta(2, 5, size=P)
    popularity = popularity / popularity.max()

    user_preferences = rng.dirichlet(np.full(C, alpha), size=U)
    user_centres = rng.random((U, 2))

    n_sessions = U * S
    base_ranks = np.empty(n_sessions, dtype=np.int16)
    top_scores = np.empty((n_sessions, M), dtype=np.float32)
    top_is_target = np.zeros((n_sessions, M), dtype=bool)
    deltas = np.empty((n_sessions, M, 3), dtype=np.float32)
    contexts = np.empty(n_sessions, dtype=np.int8)

    row = 0
    for u in range(U):
        for _ in range(S):
            contexts[row] = rng.choice(3, p=SIM_CONTEXT_PROBS)

            current_xy = np.clip(
                user_centres[u] + rng.normal(0, 0.15, size=2), 0, 1
            )
            distance = np.linalg.norm(poi_xy - current_xy, axis=1)
            proximity = np.exp(-distance / tau)
            preference = user_preferences[u, poi_cat]

            X = np.column_stack([preference, proximity, popularity])

            # Data-generating visit utility differs slightly from the recommender.
            latent_utility = (
                X @ LATENT_WEIGHTS
                + rng.normal(0, latent_sigma, size=P)
            )
            target = int(np.argmax(latent_utility))

            # Traceable feature contributions + small untraceable score noise.
            contributions = X * BASE_WEIGHTS[None, :]
            scores = (
                contributions.sum(axis=1)
                + rng.normal(0, score_sigma, size=P)
            )

            order = np.argsort(-scores, kind="stable")
            base_ranks[row] = int(np.where(order == target)[0][0]) + 1

            top = order[:M]
            top_scores[row] = scores[top]
            top_is_target[row] = (top == target)

            # Pairwise evidence convention used throughout the notebook:
            # - base winner: compare against the runner-up;
            # - any other top-M candidate: compare against the base winner.
            deltas[row, 0] = np.maximum(
                0.0, contributions[top[0]] - contributions[top[1]]
            )
            deltas[row, 1:] = np.maximum(
                0.0, contributions[top[1:]] - contributions[top[0]]
            )
            row += 1

    return {
        "base_ranks": base_ranks,
        "top_scores": top_scores,
        "top_is_target": top_is_target,
        "deltas": deltas,
        "contexts": contexts,
        "score_sigma": score_sigma,
    }


def k_from_deltas(deltas, rho):
    """Return K for every candidate plus sorted evidence and total D.

    Reproducibility convention: if D=0, set K=m (here m=3).
    """
    totals = deltas.sum(axis=2)
    sorted_delta = np.sort(deltas, axis=2)[:, :, ::-1]
    cumulative = np.cumsum(sorted_delta, axis=2)

    K = np.full(totals.shape, deltas.shape[2], dtype=np.int8)
    positive = totals > 1e-12

    ratios = np.zeros_like(cumulative, dtype=np.float32)
    ratios[positive] = cumulative[positive] / totals[positive, None]
    first = np.argmax(ratios >= rho, axis=2) + 1
    K[positive] = first[positive]
    return K, sorted_delta, totals


def recommendation_metrics(ranks):
    ranks = np.asarray(ranks)
    return {
        "Hit@5": float(np.mean(ranks <= 5)),
        "NDCG@5": float(np.mean(np.where(ranks <= 5, 1 / np.log2(ranks + 1), 0.0))),
        "MRR": float(np.mean(1 / ranks)),
    }


def evaluate_cache(cache, rho=SIM_RHO, penalty_scale=1.0,
                   budget_policy=SIM_BUDGET_POLICY, envelope_scale=1.0,
                   eta=SIM_ETA):
    base_ranks = cache["base_ranks"]
    scores = cache["top_scores"]
    is_target = cache["top_is_target"]
    contexts = cache["contexts"]
    deltas = cache["deltas"]

    K, sorted_delta, totals = k_from_deltas(deltas, rho)
    cumulative = np.cumsum(sorted_delta, axis=2)
    n = len(base_ranks)
    m = deltas.shape[2]

    B_map = np.asarray(budget_policy, dtype=int)
    B = B_map[contexts]

    # RankOnly / Fixed-3 share the same recommendation ranking.
    fixed_viol = float(np.mean(3 > B))

    # Minimal-posthoc keeps the base winner and shows the minimal rho-covering set.
    k0 = K[:, 0]
    # D=0: K=m penalizes candidates, but positive evidence units displayed = 0.
    positive_base = totals[:, 0] > 1e-12
    post_displayed = np.where(positive_base, k0, 0)
    post_units = float(post_displayed.mean())
    post_viol = float(np.mean(post_displayed > B))
    row = np.arange(n)
    post_num = cumulative[row, 0, k0 - 1]
    post_fidelity = np.zeros(n, dtype=float)
    pos = totals[:, 0] > 1e-12
    post_fidelity[pos] = post_num[pos] / totals[pos, 0]

    # CL-MEE parameters are context-indexed.
    lambdas = np.asarray(SIM_LAMBDAS) * penalty_scale
    epsilons = np.asarray(SIM_EPSILONS) * envelope_scale
    lam = lambdas[contexts]
    eps = epsilons[contexts]

    eligible = scores >= (scores[:, [0]] - eps[:, None])
    burden = np.maximum(0, K - B[:, None])
    adjusted = scores - lam[:, None] * burden - eta * K / m
    adjusted = np.where(eligible, adjusted, -np.inf)
    chosen_pos = np.argmax(adjusted, axis=1)
    chosen_rank = chosen_pos + 1
    chosen_is_target = is_target[row, chosen_pos]

    # Move the selected candidate to rank 1 and retain the order of all others.
    cl_rank = base_ranks.astype(int).copy()
    cl_rank[chosen_is_target] = 1
    shifts_down = (~chosen_is_target) & (chosen_rank > base_ranks)
    cl_rank[shifts_down] += 1

    chosen_K = K[row, chosen_pos]
    # Hard display truncation is separate from the soft penalty.
    chosen_total = totals[row, chosen_pos]
    displayed_units = np.where(chosen_total > 1e-12, np.minimum(chosen_K, B), 0)

    chosen_cumulative = cumulative[row, chosen_pos]
    chosen_num = np.zeros(n, dtype=float)
    has_display = displayed_units > 0
    chosen_num[has_display] = chosen_cumulative[has_display, displayed_units[has_display] - 1]
    cl_fidelity = np.zeros(n, dtype=float)
    pos = chosen_total > 1e-12
    cl_fidelity[pos] = chosen_num[pos] / chosen_total[pos]

    return {
        "RankOnly": recommendation_metrics(base_ranks),
        "CL-MEE": recommendation_metrics(cl_rank),
        "Fixed-3_units": 3.0,
        "Fixed-3_viol": fixed_viol,
        "Fixed-3_fidelity": float(np.mean(totals[:, 0] > 1e-12)),
        "Minimal-posthoc_units": post_units,
        "Minimal-posthoc_viol": post_viol,
        "Minimal-posthoc_fidelity": float(post_fidelity.mean()),
        "CL-MEE_units": float(displayed_units.mean()),
        "CL-MEE_viol": float(np.mean(displayed_units > B)),
        "CL-MEE_overbudget_before_display": float(np.mean(chosen_K > B)),
        "CL-MEE_D0": float(np.mean(chosen_total <= 1e-12)),
        "CL-MEE_fidelity": float(cl_fidelity.mean()),
        "CL-MEE_rerank": float(np.mean(chosen_pos != 0)),
    }


In [95]:

# Generate the 10 fixed synthetic seeds once; sensitivity checks reuse them.
sim_caches = [generate_seed_cache(seed) for seed in SIM_SEEDS]
sim_results = [evaluate_cache(cache) for cache in sim_caches]


def mean_sd(values):
    values = np.asarray(values, dtype=float)
    return values.mean(), values.std(ddof=1)

rows = []
for method in ["RankOnly", "Fixed-3", "Minimal-posthoc", "CL-MEE"]:
    ranking_source = "CL-MEE" if method == "CL-MEE" else "RankOnly"
    hit_m, hit_sd = mean_sd([r[ranking_source]["Hit@5"] for r in sim_results])
    ndcg_m, ndcg_sd = mean_sd([r[ranking_source]["NDCG@5"] for r in sim_results])
    mrr_m, mrr_sd = mean_sd([r[ranking_source]["MRR"] for r in sim_results])

    if method == "RankOnly":
        units = viol = fidelity = np.nan
    elif method == "Fixed-3":
        units = np.mean([r["Fixed-3_units"] for r in sim_results])
        viol = np.mean([r["Fixed-3_viol"] for r in sim_results])
        fidelity = np.mean([r["Fixed-3_fidelity"] for r in sim_results])
    elif method == "Minimal-posthoc":
        units = np.mean([r["Minimal-posthoc_units"] for r in sim_results])
        viol = np.mean([r["Minimal-posthoc_viol"] for r in sim_results])
        fidelity = np.mean([r["Minimal-posthoc_fidelity"] for r in sim_results])
    else:
        units = np.mean([r["CL-MEE_units"] for r in sim_results])
        viol = np.mean([r["CL-MEE_viol"] for r in sim_results])
        fidelity = np.mean([r["CL-MEE_fidelity"] for r in sim_results])

    rows.append({
        "Method": method,
        "Hit@5 mean": hit_m, "Hit@5 SD": hit_sd,
        "NDCG@5 mean": ndcg_m, "NDCG@5 SD": ndcg_sd,
        "MRR mean": mrr_m, "MRR SD": mrr_sd,
        "Units": units, "Viol.": viol, "F_model": fidelity,
    })

sim_table2 = pd.DataFrame(rows)
sim_table2


,Method,Hit@5 mean,Hit@5 SD,NDCG@5 mean,NDCG@5 SD,MRR mean,MRR SD,Units,Viol.,F_model
0,RankOnly,0.199125,0.008728,0.131031,0.006611,0.138927,0.006031,NaN,NaN,NaN
1,Fixed-3,0.199125,0.008728,0.131031,0.006611,0.138927,0.006031,3.00000,0.652000,0.988417
2,Minimal-posthoc,0.199125,0.008728,0.131031,0.006611,0.138927,0.006031,1.24075,0.064042,0.973209
3,CL-MEE,0.199125,0.008728,0.130609,0.006575,0.138358,0.006062,1.18225,0.000000,0.971935


In [96]:

# Seed-level inference used in the paper discussion.
base_ndcg_seed = np.array([r["RankOnly"]["NDCG@5"] for r in sim_results])
clmee_ndcg_seed = np.array([r["CL-MEE"]["NDCG@5"] for r in sim_results])
post_f_seed = np.array([r["Minimal-posthoc_fidelity"] for r in sim_results])
clmee_f_seed = np.array([r["CL-MEE_fidelity"] for r in sim_results])

ndcg_test = stats.ttest_rel(clmee_ndcg_seed, base_ndcg_seed)
fidelity_test = stats.ttest_rel(clmee_f_seed, post_f_seed)

simulation_summary = {
    "Baseline NDCG@5": base_ndcg_seed.mean(),
    "CL-MEE NDCG@5": clmee_ndcg_seed.mean(),
    "NDCG retained (%)": 100 * clmee_ndcg_seed.mean() / base_ndcg_seed.mean(),
    "Mean NDCG difference": (clmee_ndcg_seed - base_ndcg_seed).mean(),
    "Paired t-test NDCG p": ndcg_test.pvalue,
    "Posthoc F_model": post_f_seed.mean(),
    "CL-MEE F_model": clmee_f_seed.mean(),
    "Paired t-test fidelity p": fidelity_test.pvalue,
    "CL-MEE units": np.mean([r["CL-MEE_units"] for r in sim_results]),
}

pd.Series(simulation_summary)


Baseline NDCG@5              0.131031
CL-MEE NDCG@5                0.130609
NDCG retained (%)           99.677964
Mean NDCG difference        -0.000422
Paired t-test NDCG p         0.085140
Posthoc F_model              0.973209
CL-MEE F_model               0.971935
Paired t-test fidelity p     0.122154
CL-MEE units                 1.182250
dtype: float64

## 2. One-factor-at-a-time sensitivity analysis (Table 3)

In [97]:

def aggregate_sensitivity(caches, **kwargs):
    r = [evaluate_cache(cache, **kwargs) for cache in caches]
    base_ndcg = np.mean([x["RankOnly"]["NDCG@5"] for x in r])
    cl_ndcg = np.mean([x["CL-MEE"]["NDCG@5"] for x in r])
    return {
        "NDCG retained (%)": 100 * cl_ndcg / base_ndcg,
        "F_model": np.mean([x["CL-MEE_fidelity"] for x in r]),
        "Units": np.mean([x["CL-MEE_units"] for x in r]),
        "Re-rank rate": np.mean([x["CL-MEE_rerank"] for x in r]),
    }

sensitivity_rows = []

for value in [0.70, 0.80, 0.90]:
    sensitivity_rows.append({"Factor": "Evidence coverage rho", "Value": value,
                             **aggregate_sensitivity(sim_caches, rho=value)})

for value in [0.5, 1.0, 1.5]:
    sensitivity_rows.append({"Factor": "Penalty scale", "Value": value,
                             **aggregate_sensitivity(sim_caches, penalty_scale=value)})

for value in [(1,1,1), (3,2,1), (3,3,3)]:
    sensitivity_rows.append({"Factor": "Budget policy", "Value": str(value),
                             **aggregate_sensitivity(sim_caches, budget_policy=value)})

for value in [0.5, 1.0, 1.5]:
    sensitivity_rows.append({"Factor": "Utility envelope scale", "Value": value,
                             **aggregate_sensitivity(sim_caches, envelope_scale=value)})

# Score-noise sensitivity requires regenerated caches because it changes the base score.
for value in [0.010, 0.015, 0.030]:
    noise_caches = [generate_seed_cache(seed, score_sigma=value) for seed in SIM_SEEDS]
    sensitivity_rows.append({"Factor": "Score noise sigma", "Value": value,
                             **aggregate_sensitivity(noise_caches)})

sensitivity_detailed = pd.DataFrame(sensitivity_rows)
sensitivity_detailed


,Factor,Value,NDCG retained (%),F_model,Units,Re-rank rate
0,Evidence coverage rho,0.7,99.822961,0.952989,1.121333,0.037333
1,Evidence coverage rho,0.8,99.677964,0.971935,1.182250,0.051417
2,Evidence coverage rho,0.9,99.541294,0.982570,1.239042,0.063750
3,Penalty scale,0.5,99.923218,0.965584,1.181833,0.034333
4,Penalty scale,1.0,99.677964,0.971935,1.182250,0.051417
5,Penalty scale,1.5,99.620151,0.974592,1.182292,0.058542
6,Budget policy,"(1, 1, 1)",99.587485,0.919888,0.995417,0.086917
7,Budget policy,"(3, 2, 1)",99.677964,0.971935,1.182250,0.051417
8,Budget policy,"(3, 3, 3)",100.023475,0.975203,1.239042,0.006917
9,Utility envelope scale,0.5,99.889764,0.967535,1.182125,0.039833


In [98]:

# Compact ranges used directly in Table 3 of the paper.
summary_rows = []
for factor, group in sensitivity_detailed.groupby("Factor", sort=False):
    summary_rows.append({
        "Factor": factor,
        "NDCG retained range (%)": f"{group['NDCG retained (%)'].min():.2f}-{group['NDCG retained (%)'].max():.2f}",
        "F_model range": f"{group['F_model'].min():.3f}-{group['F_model'].max():.3f}",
        "Units range": f"{group['Units'].min():.2f}-{group['Units'].max():.2f}",
    })

sim_table3 = pd.DataFrame(summary_rows)
sim_table3


,Factor,NDCG retained range (%),F_model range,Units range
0,Evidence coverage rho,99.54-99.82,0.953-0.983,1.12-1.24
1,Penalty scale,99.62-99.92,0.966-0.975,1.18-1.18
2,Budget policy,99.59-100.02,0.920-0.975,1.00-1.24
3,Utility envelope scale,99.68-99.89,0.968-0.972,1.18-1.18
4,Score noise sigma,99.66-99.70,0.962-0.975,1.18-1.18


## 3. Foursquare NYC offline evaluation (Table 4)

The following block preserves the real-data pipeline from the supplied notebook. The 80/10/10 split is **chronological within each user history**. The five non-negative scoring weights are learned on up to 5,000 validation transitions using pairwise logistic loss with L2 regularization, then normalized to sum to one.

In [99]:
# Load the public Foursquare NYC check-in file
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH.resolve()}")

columns = [
    "user_id",
    "venue_id",
    "category_id",
    "category_name",
    "latitude",
    "longitude",
    "timezone_offset",
    "utc_time"
]

df = pd.read_csv(
    DATASET_PATH,
    sep="\t",
    names=columns,
    encoding="latin-1"
)

df.head()

,user_id,venue_id,category_id,category_name,latitude,longitude,timezone_offset,utc_time
0,470,49bbd6c0f964a520f4531fe3,4bf58dd8d48988d127951735,Arts & Crafts Store,40.719810,-74.002581,-240,Tue Apr 03 18:00:09 +0000 2012
1,979,4a43c0aef964a520c6a61fe3,4bf58dd8d48988d1df941735,Bridge,40.606800,-74.044170,-240,Tue Apr 03 18:00:25 +0000 2012
2,69,4c5cc7b485a1e21e00d35711,4bf58dd8d48988d103941735,Home (private),40.716162,-73.883070,-240,Tue Apr 03 18:02:24 +0000 2012
3,395,4bc7086715a7ef3bef9878da,4bf58dd8d48988d104941735,Medical Center,40.745164,-73.982519,-240,Tue Apr 03 18:02:41 +0000 2012
4,87,4cf2c5321d18a143951b5cec,4bf58dd8d48988d1cb941735,Food Truck,40.740104,-73.989658,-240,Tue Apr 03 18:03:00 +0000 2012


In [100]:
print("Check-ins:", len(df))
print("Users:", df["user_id"].nunique())
print("POIs:", df["venue_id"].nunique())
print("Categories:", df["category_id"].nunique())

Check-ins: 227428
Users: 1083
POIs: 38333
Categories: 400


In [101]:
df["utc_time"] = pd.to_datetime(
    df["utc_time"],
    format="%a %b %d %H:%M:%S %z %Y"
)

print("First date:", df["utc_time"].min())
print("Last date:", df["utc_time"].max())
print("\nColumn type:")
print(df["utc_time"].dtype)

First date: 2012-04-03 18:00:09+00:00
Last date: 2013-02-16 02:35:36+00:00

Column type:
datetime64[us, UTC]


In [102]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
user_id            0
venue_id           0
category_id        0
category_name      0
latitude           0
longitude          0
timezone_offset    0
utc_time           0
dtype: int64


In [103]:
df = df.sort_values(
    ["user_id", "utc_time", "venue_id"], kind="stable"
).reset_index(drop=True)

print("Sorted successfully.")
print(df.head())

Sorted successfully.
   user_id                  venue_id               category_id  \
0        1  4abc1f51f964a520798620e3  4bf58dd8d48988d1ce941735   
1        1  4d4ac10da0ef54814b6ffff6  4bf58dd8d48988d157941735   
2        1  4db44994cda1c57c82583709  4bf58dd8d48988d1f1931735   
3        1  4a541923f964a52008b31fe3  4bf58dd8d48988d14e941735   
4        1  40f1d480f964a5205b0a1fe3  4bf58dd8d48988d143941735   

           category_name   latitude  longitude  timezone_offset  \
0     Seafood Restaurant  40.781558 -73.975792             -240   
1    American Restaurant  40.784018 -73.974524             -240   
2  General Entertainment  40.739398 -73.993210             -240   
3    American Restaurant  40.785677 -73.976498             -240   
4         Breakfast Spot  40.719929 -74.008532             -240   

                   utc_time  
0 2012-04-04 23:31:31+00:00  
1 2012-04-07 17:42:24+00:00  
2 2012-04-08 18:20:29+00:00  
3 2012-04-08 20:02:10+00:00  
4 2012-04-09 16:20:52+00:00  

In [104]:
user_counts = df.groupby("user_id").size().sort_values(ascending=False)

print(user_counts.head(10))

user_id
293    2697
185    2079
354    2061
315    1682
84     1376
349    1369
384    1116
974    1107
768    1096
445     952
dtype: int64


In [105]:
top_user = user_counts.index[0]

print("Selected user:", top_user)
print("Number of check-ins:", user_counts.iloc[0])

df[df["user_id"] == top_user][
    [
        "utc_time",
        "venue_id",
        "category_name",
        "latitude",
        "longitude"
    ]
].head(15)

Selected user: 293
Number of check-ins: 2697


,utc_time,venue_id,category_name,latitude,longitude
57863,2012-04-03 23:56:43+00:00,4ae8fd76f964a520e1b321e3,Subway,40.725577,-74.004252
57864,2012-04-04 00:19:13+00:00,4d5bedbb5d153704613a6ce7,Other Great Outdoors,40.714310,-73.949394
57865,2012-04-04 00:19:44+00:00,41156d00f964a520f40b1fe3,Bar,40.713624,-73.949710
57866,2012-04-04 00:21:12+00:00,4d61b919865a224bef72ba85,Smoke Shop,40.715951,-73.949595
57867,2012-04-04 04:53:06+00:00,4c97edbdf419a09395806b88,Neighborhood,40.650169,-73.949575
57868,2012-04-04 04:53:36+00:00,4d5bedbb5d153704613a6ce7,Other Great Outdoors,40.714310,-73.949394
57869,2012-04-04 05:08:28+00:00,4e6eab51d1647b1137a3a2d0,Bus Station,40.704168,-73.948189
57870,2012-04-04 15:38:43+00:00,4d4b5ceb8e948cfa35fcef48,Home (private),40.693212,-73.930104
57871,2012-04-04 16:16:45+00:00,4e5577f445dd0a4826e6ab3e,Home (private),40.693212,-73.930104
57872,2012-04-04 16:18:34+00:00,4d2f5ae2940137044fa1eeda,General Travel,40.689003,-73.929085


In [106]:
# Position of each check-in inside each user's chronological history
group_size = df.groupby("user_id")["user_id"].transform("size")
group_position = df.groupby("user_id").cumcount()

# Relative chronological position
fraction = (group_position + 1) / group_size

# Per-user chronological 80% train, 10% validation, 10% test
df["split"] = np.where(
    fraction <= 0.8,
    "train",
    np.where(fraction <= 0.9, "validation", "test")
)

print(df["split"].value_counts())

split
train         181492
test           23244
validation     22692
Name: count, dtype: int64


In [107]:
train = df[df["split"] == "train"].copy()
validation = df[df["split"] == "validation"].copy()
test = df[df["split"] == "test"].copy()

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))
print("Total:", len(train) + len(validation) + len(test))

Train: 181492
Validation: 22692
Test: 23244
Total: 227428


In [108]:
print("Unique category IDs:", df["category_id"].nunique())
print("Unique category names:", df["category_name"].nunique())

Unique category IDs: 400
Unique category names: 251


In [109]:
# Previous check-in context for each user
df["prev_lat"] = df.groupby("user_id")["latitude"].shift(1)
df["prev_lon"] = df.groupby("user_id")["longitude"].shift(1)
df["prev_category"] = df.groupby("user_id")["category_name"].shift(1)

# Convert UTC time to local time using the timezone offset
df["local_time"] = (
    df["utc_time"]
    + pd.to_timedelta(df["timezone_offset"], unit="m")
)

df["prev_local_time"] = (
    df.groupby("user_id")["local_time"].shift(1)
)

print(df[
    [
        "user_id",
        "utc_time",
        "category_name",
        "prev_category",
        "prev_lat",
        "prev_lon"
    ]
].head(10))

   user_id                  utc_time          category_name  \
0        1 2012-04-04 23:31:31+00:00     Seafood Restaurant   
1        1 2012-04-07 17:42:24+00:00    American Restaurant   
2        1 2012-04-08 18:20:29+00:00  General Entertainment   
3        1 2012-04-08 20:02:10+00:00    American Restaurant   
4        1 2012-04-09 16:20:52+00:00         Breakfast Spot   
5        1 2012-04-10 00:24:31+00:00             Steakhouse   
6        1 2012-04-10 03:36:56+00:00                    Bar   
7        1 2012-04-10 16:21:48+00:00     Mexican Restaurant   
8        1 2012-04-12 17:19:21+00:00            Coffee Shop   
9        1 2012-04-13 15:41:41+00:00     Mexican Restaurant   

           prev_category   prev_lat   prev_lon  
0                    NaN        NaN        NaN  
1     Seafood Restaurant  40.781558 -73.975792  
2    American Restaurant  40.784018 -73.974524  
3  General Entertainment  40.739398 -73.993210  
4    American Restaurant  40.785677 -73.976498  
5         Br

In [110]:
venue_catalog = (
    train.groupby("venue_id", as_index=False)
    .agg(
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
        category_name=("category_name", "first"),
        category_id=("category_id", "first")
    )
)

train_venues = set(venue_catalog["venue_id"])

print("POIs in complete dataset:", df["venue_id"].nunique())
print("POIs seen in training:", len(train_venues))

print(
    "Percentage of test visits whose POI was already seen in training:",
    round(test["venue_id"].isin(train_venues).mean() * 100, 2),
    "%"
)

POIs in complete dataset: 38333
POIs seen in training: 34117
Percentage of test visits whose POI was already seen in training: 87.0 %


In [111]:
validation_eligible = df[
    (df["split"] == "validation")
    & (df["venue_id"].isin(train_venues))
    & (df["prev_lat"].notna())
    & (df["prev_local_time"].notna())
].copy()

test_eligible = df[
    (df["split"] == "test")
    & (df["venue_id"].isin(train_venues))
    & (df["prev_lat"].notna())
    & (df["prev_local_time"].notna())
].copy()

print("Eligible validation transitions:", len(validation_eligible))
print("Eligible test transitions:", len(test_eligible))

Eligible validation transitions: 20211
Eligible test transitions: 20223


In [112]:
# =========================
# 1. TRAINING POI CATALOG
# =========================

venue_meta = (
    train.groupby("venue_id", as_index=False)
    .agg(
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
        category_name=("category_name", "first"),
        category_id=("category_id", "first")
    )
    .reset_index(drop=True)
)

venue_to_idx = pd.Series(
    venue_meta.index.values,
    index=venue_meta["venue_id"]
).to_dict()

print("Training POIs:", len(venue_meta))

Training POIs: 34117


In [113]:
# =========================
# 2. CATEGORY MAPPING
# =========================

categories = pd.Index(
    sorted(train["category_name"].astype(str).unique())
)

category_to_idx = {
    category: i
    for i, category in enumerate(categories)
}

venue_meta["cat_idx"] = (
    venue_meta["category_name"]
    .astype(str)
    .map(category_to_idx)
    .astype(int)
)

print("Training category names:", len(categories))
print(categories[:10])

Training category names: 251
Index(['Afghan Restaurant', 'African Restaurant', 'Airport',
       'American Restaurant', 'Animal Shelter', 'Antique Shop', 'Aquarium',
       'Arcade', 'Arepa Restaurant', 'Argentinian Restaurant'],
      dtype='str')


In [114]:
# =========================
# 3. POI POPULARITY
# =========================

popularity = train.groupby("venue_id").size()

venue_meta["pop_count"] = (
    venue_meta["venue_id"]
    .map(popularity)
    .fillna(0)
    .astype(int)
)

venue_meta["popularity_feature"] = (
    np.log1p(venue_meta["pop_count"])
    / np.log1p(venue_meta["pop_count"].max())
)

venue_meta[
    [
        "venue_id",
        "category_name",
        "pop_count",
        "popularity_feature"
    ]
].sort_values(
    "pop_count",
    ascending=False
).head(10)

,venue_id,category_name,pop_count,popularity_feature
1261,42911d00f964a520f5231fe3,Train Station,927,1.000000
1239,42829c80f964a5206a221fe3,Train Station,703,0.959571
1445,43a52546f964a520532c1fe3,Airport,520,0.915516
2163,4840fe6bf964a52030501fe3,Bus Station,443,0.892111
6407,4ace6c89f964a52078d020e3,Airport,413,0.881873
4526,4a737bf8f964a52091dc1fe3,Subway,395,0.875368
871,40b68100f964a5207d001fe3,Park,360,0.861825
2264,49b7ed6df964a52030531fe3,Plaza,353,0.858959
3813,4a4821f5f964a52095aa1fe3,Road,326,0.847349
3904,4a513b17f964a520d2b01fe3,Train Station,325,0.846900


In [115]:
# =========================
# 4. USER CATEGORY PREFERENCE
# Clean user indexing
# =========================

users = pd.Index(
    sorted(df["user_id"].unique())
)

user_to_idx = {
    user_id: i
    for i, user_id in enumerate(users)
}

n_users = len(users)
n_categories = len(categories)

user_category = np.zeros(
    (n_users, n_categories),
    dtype=np.float32
)

temp_train = train.copy()

temp_train["user_idx"] = (
    temp_train["user_id"]
    .map(user_to_idx)
)

temp_train["cat_idx"] = (
    temp_train["category_name"]
    .astype(str)
    .map(category_to_idx)
)

category_counts = (
    temp_train.groupby(
        ["user_idx", "cat_idx"]
    ).size()
)

for (user_idx, cat_idx), count in category_counts.items():
    user_category[
        int(user_idx),
        int(cat_idx)
    ] = count

# Smoothing
user_category = (
    user_category + 0.5
) / (
    user_category.sum(axis=1, keepdims=True)
    + 0.5 * n_categories
)

# Normalize each user's strongest category preference to 1
user_category_norm = (
    user_category
    / (
        user_category.max(
            axis=1,
            keepdims=True
        ) + 1e-12
    )
)

print("Users:", n_users)
print("Categories:", n_categories)
print("User-category matrix shape:", user_category_norm.shape)

Users: 1083
Categories: 251
User-category matrix shape: (1083, 251)


In [116]:
# ==================================
# 5. TEMPORAL CATEGORY COMPATIBILITY
# ==================================

train2 = train.copy()

# Convert UTC time to local time
train2["local_time"] = (
    train2["utc_time"]
    + pd.to_timedelta(
        train2["timezone_offset"],
        unit="m"
    )
)

# Six periods of 4 hours each
train2["hbin"] = (
    train2["local_time"].dt.hour // 4
).astype(int)

# Start with smoothing value 0.5
temporal = np.ones(
    (6, n_categories),
    dtype=np.float32
) * 0.5

train2["cat_idx"] = (
    train2["category_name"]
    .astype(str)
    .map(category_to_idx)
)

temporal_counts = (
    train2.groupby(
        ["hbin", "cat_idx"]
    ).size()
)

for (hour_bin, cat_idx), count in temporal_counts.items():
    temporal[
        int(hour_bin),
        int(cat_idx)
    ] += count

# Convert counts to probabilities
temporal = (
    temporal
    / (
        temporal.sum(
            axis=0,
            keepdims=True
        ) + 1e-12
    )
)

# Normalize strongest period for each category to 1
temporal_norm = (
    temporal
    / (
        temporal.max(
            axis=0,
            keepdims=True
        ) + 1e-12
    )
)

print("Temporal matrix shape:", temporal_norm.shape)
print("Hour bins:", sorted(train2["hbin"].unique()))

Temporal matrix shape: (6, 251)
Hour bins: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [117]:
def show_temporal_pattern(category):
    cat_idx = category_to_idx[category]

    result = pd.DataFrame({
        "Period": [
            "00:00-03:59",
            "04:00-07:59",
            "08:00-11:59",
            "12:00-15:59",
            "16:00-19:59",
            "20:00-23:59"
        ],
        "Temporal compatibility": temporal_norm[:, cat_idx]
    })

    return result


show_temporal_pattern("Breakfast Spot")

,Period,Temporal compatibility
0,00:00-03:59,0.118367
1,04:00-07:59,0.534694
2,08:00-11:59,1.000000
3,12:00-15:59,0.714286
4,16:00-19:59,0.436735
5,20:00-23:59,0.346939


In [118]:
# ==================================
# 6. CATEGORY-TRANSITION COMPATIBILITY
# ==================================

train_transition = train.copy()

# Ensure chronological order
train_transition = train_transition.sort_values(
    ["user_id", "utc_time", "venue_id"], kind="stable"
).reset_index(drop=True)

# Previous category within each user's TRAINING history
train_transition["prev_category"] = (
    train_transition.groupby("user_id")["category_name"]
    .shift(1)
)

# Remove rows with no previous category
transition_rows = train_transition.dropna(
    subset=["prev_category"]
).copy()

# Map category names to indices
transition_rows["prev_cat_idx"] = (
    transition_rows["prev_category"]
    .astype(str)
    .map(category_to_idx)
)

transition_rows["current_cat_idx"] = (
    transition_rows["category_name"]
    .astype(str)
    .map(category_to_idx)
)

# Remove anything that could not be mapped
transition_rows = transition_rows.dropna(
    subset=["prev_cat_idx", "current_cat_idx"]
)

# Start with smoothing
transition_matrix = np.ones(
    (n_categories, n_categories),
    dtype=np.float32
) * 0.5

# Count transitions
transition_counts = (
    transition_rows.groupby(
        ["prev_cat_idx", "current_cat_idx"]
    ).size()
)

for (prev_idx, current_idx), count in transition_counts.items():
    transition_matrix[
        int(prev_idx),
        int(current_idx)
    ] += count

# Convert each row to probabilities
transition_matrix = (
    transition_matrix
    / (
        transition_matrix.sum(
            axis=1,
            keepdims=True
        ) + 1e-12
    )
)

# Normalize strongest transition from each category to 1
transition_norm = (
    transition_matrix
    / (
        transition_matrix.max(
            axis=1,
            keepdims=True
        ) + 1e-12
    )
)

print("Transition matrix shape:", transition_norm.shape)
print("Training transitions used:", len(transition_rows))

Transition matrix shape: (251, 251)
Training transitions used: 180409


In [119]:
def top_next_categories(previous_category, top_n=10):
    prev_idx = category_to_idx[previous_category]

    scores = transition_norm[prev_idx]

    order = np.argsort(scores)[::-1][:top_n]

    return pd.DataFrame({
        "Previous category": [previous_category] * top_n,
        "Next category": categories[order],
        "Transition compatibility": scores[order]
    })


top_next_categories("Subway", 10)

,Previous category,Next category,Transition compatibility
0,Subway,Subway,1.000000
1,Subway,Train Station,0.246633
2,Subway,Home (private),0.149749
3,Subway,Office,0.128040
4,Subway,Neighborhood,0.111156
5,Subway,Bus Station,0.088241
6,Subway,Deli / Bodega,0.078593
7,Subway,Food & Drink Shop,0.060101
8,Subway,Coffee Shop,0.058090
9,Subway,Bar,0.056884


In [120]:
# ==================================
# 7. GEOGRAPHIC PROXIMITY
# ==================================

# Keep only TRAIN rows that have a previous location
train_steps = df[
    (df["split"] == "train")
    & (df["prev_lat"].notna())
    & (df["prev_lon"].notna())
].copy()

# Convert coordinates to radians
lat1 = np.deg2rad(train_steps["prev_lat"].to_numpy())
lon1 = np.deg2rad(train_steps["prev_lon"].to_numpy())

lat2 = np.deg2rad(train_steps["latitude"].to_numpy())
lon2 = np.deg2rad(train_steps["longitude"].to_numpy())

# Haversine distance
a = (
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin((lon2 - lon1) / 2) ** 2
)

distance_km = (
    6371
    * 2
    * np.arcsin(
        np.minimum(1, np.sqrt(a))
    )
)

# Median real movement distance in TRAIN
TAU = float(np.median(distance_km))

print("Training transitions:", len(distance_km))
print("Median movement distance TAU:", TAU, "km")

print("\nDistance statistics:")
print(pd.Series(distance_km).describe())

Training transitions: 180409
Median movement distance TAU: 1.5114327113609778 km

Distance statistics:
count    180409.000000
mean          4.041260
std           5.620183
min           0.000000
25%           0.277950
50%           1.511433
75%           5.491445
max          44.207262
dtype: float64


In [121]:
def proximity_score(distance_km):
    return np.exp(-distance_km / TAU)

example_distances = np.array([
    0,
    0.5,
    1,
    1.5,
    3,
    5,
    10
])

pd.DataFrame({
    "Distance_km": example_distances,
    "Proximity_score": proximity_score(example_distances)
})

,Distance_km,Proximity_score
0,0.0,1.000000
1,0.5,0.718340
2,1.0,0.516013
3,1.5,0.370673
4,3.0,0.137398
5,5.0,0.036585
6,10.0,0.001338


In [122]:
# ==================================
# 8. PREPARE POI ARRAYS
# ==================================

from sklearn.neighbors import BallTree

# Arrays for fast access
v_lat = venue_meta["latitude"].to_numpy()
v_lon = venue_meta["longitude"].to_numpy()
v_cat = venue_meta["cat_idx"].to_numpy()
v_pop = venue_meta["popularity_feature"].to_numpy(dtype=np.float32)

# Geographic index
coords = np.deg2rad(
    venue_meta[["latitude", "longitude"]].to_numpy()
)

tree = BallTree(
    coords,
    metric="haversine"
)

print("POIs indexed:", len(venue_meta))
print("BallTree created successfully.")

POIs indexed: 34117
BallTree created successfully.


In [123]:
# ==================================
# 9. FIVE FEATURES FOR A CANDIDATE POI
# ==================================

def event_features(row, candidate_indices):

    candidate_indices = np.asarray(candidate_indices)

    # Candidate categories
    cat_indices = v_cat[candidate_indices]

    # 1. User-category preference
    user_idx = user_to_idx[row["user_id"]]

    preference = user_category_norm[
        user_idx,
        cat_indices
    ]

    # 2. Geographic proximity
    lat1 = np.deg2rad(float(row["prev_lat"]))
    lon1 = np.deg2rad(float(row["prev_lon"]))

    lat2 = np.deg2rad(v_lat[candidate_indices])
    lon2 = np.deg2rad(v_lon[candidate_indices])

    a = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((lon2 - lon1) / 2) ** 2
    )

    distance = (
        6371
        * 2
        * np.arcsin(
            np.minimum(1, np.sqrt(a))
        )
    )

    proximity = np.exp(-distance / TAU)

    # 3. Popularity
    popularity = v_pop[candidate_indices]

    # 4. Temporal compatibility
    hour_bin = int(
        row["prev_local_time"].hour // 4
    )

    temporal_fit = temporal_norm[
        hour_bin,
        cat_indices
    ]

    # 5. Category-transition compatibility
    previous_category = str(row["prev_category"])

    if previous_category in category_to_idx:

        prev_cat_idx = category_to_idx[
            previous_category
        ]

        transition_fit = transition_norm[
            prev_cat_idx,
            cat_indices
        ]

    else:
        transition_fit = np.zeros(
            len(candidate_indices),
            dtype=np.float32
        )

    return np.column_stack([
        preference,
        proximity,
        popularity,
        temporal_fit,
        transition_fit
    ]).astype(np.float32)

In [124]:
# ==================================
# 10. TEST ONE REAL VALIDATION EVENT
# ==================================

sample = validation_eligible.iloc[0]

target_idx = venue_to_idx[
    sample["venue_id"]
]

features = event_features(
    sample,
    [target_idx]
)[0]

result = pd.DataFrame({
    "Feature": [
        "User preference",
        "Proximity",
        "Popularity",
        "Temporal compatibility",
        "Transition compatibility"
    ],
    "Value": features
})

print("User:", sample["user_id"])
print("Actual visited POI:", sample["venue_id"])
print("Category:", sample["category_name"])
print("Previous category:", sample["prev_category"])
print()

result

User: 1
Actual visited POI: 4ae8f5ddf964a520b4b321e3
Category: Salon / Barbershop
Previous category: American Restaurant



,Feature,Value
0,User preference,0.103448
1,Proximity,0.007205
2,Popularity,0.160780
3,Temporal compatibility,0.317647
4,Transition compatibility,0.055772


In [125]:
from scipy.optimize import minimize

# ==================================
# 11. LEARN BASE RECOMMENDER WEIGHTS
#     USING VALIDATION ONLY
# Pairwise logistic loss; L2 regularization is applied in the next cell.
# ==================================

SEED = 20260919
rng = np.random.default_rng(SEED)

# Popularity-based sampling probability
pop_prob = np.sqrt(
    venue_meta["pop_count"].to_numpy(dtype=float) + 1
)
pop_prob = pop_prob / pop_prob.sum()

# Venues grouped by category
cat_to_venues = {
    ci: venue_meta.index[
        venue_meta["cat_idx"] == ci
    ].to_numpy()
    for ci in range(n_categories)
}

# Use up to 5000 validation transitions (paper protocol)
val_sample = validation_eligible.sample(
    n=min(5000, len(validation_eligible)),
    random_state=42
).reset_index(drop=True)

# Find nearby POIs for each validation event
query_coords = np.deg2rad(
    val_sample[["prev_lat", "prev_lon"]].to_numpy()
)

_, near100 = tree.query(
    query_coords,
    k=100
)

diffs = []

for j, row in val_sample.iterrows():

    # Actual visited POI
    target = venue_to_idx[row["venue_id"]]

    target_cat = v_cat[target]

    negatives = []

    # 1. Same-category negatives
    same_category = cat_to_venues[target_cat]
    same_category = same_category[
        same_category != target
    ]

    if len(same_category) > 0:
        chosen = rng.choice(
            same_category,
            size=min(10, len(same_category)),
            replace=False
        )

        negatives.extend(
            [int(x) for x in chosen]
        )

    # 2. Nearby negatives
    for x in near100[j]:

        x = int(x)

        if (
            x != target
            and x not in negatives
        ):
            negatives.append(x)

        if len(negatives) >= 15:
            break

    # 3. Popularity-focused negatives
    while len(negatives) < 20:

        x = int(
            rng.choice(
                len(venue_meta),
                p=pop_prob
            )
        )

        if (
            x != target
            and x not in negatives
        ):
            negatives.append(x)

    # Features of actual visited POI
    positive_features = event_features(
        row,
        np.array([target])
    )[0]

    # Features of negative POIs
    negative_features = event_features(
        row,
        np.array(negatives[:20])
    )

    # Pairwise feature differences
    diffs.append(
        positive_features[None, :]
        - negative_features
    )

diffs = np.vstack(diffs)

print("Pairwise comparisons:", len(diffs))
print("Feature dimensions:", diffs.shape[1])

Pairwise comparisons: 100000
Feature dimensions: 5


In [126]:
# ==================================
# 12. OPTIMIZE NON-NEGATIVE WEIGHTS
# ==================================

ALPHA = 0.10

def objective(w):

    margins = diffs.dot(w)

    pairwise_loss = np.logaddexp(
        0,
        -margins
    ).mean()

    regularization = ALPHA * np.sum(w ** 2)

    return pairwise_loss + regularization


optimization = minimize(
    objective,
    x0=np.ones(5) * 0.5,
    bounds=[(0, None)] * 5,
    method="L-BFGS-B"
)

weights = optimization.x

weights = weights / weights.sum()

feature_names = [
    "User preference",
    "Proximity",
    "Popularity",
    "Temporal compatibility",
    "Transition compatibility"
]

weight_table = pd.DataFrame({
    "Feature": feature_names,
    "Weight": weights
})

print("Optimization successful:",
      optimization.success)

weight_table

Optimization successful: True


,Feature,Weight
0,User preference,0.306132
1,Proximity,0.223335
2,Popularity,0.408611
3,Temporal compatibility,0.022750
4,Transition compatibility,0.039173


In [127]:
test_eligible = test_eligible.reset_index(drop=True)

print("Test transitions:", len(test_eligible))
print("Index starts at:", test_eligible.index.min())
print("Index ends at:", test_eligible.index.max())

Test transitions: 20223
Index starts at: 0
Index ends at: 20222


In [128]:
# ==================================
# 13. BUILD TEST CANDIDATE SETS
# ==================================

SEED = 20260919

N_NEAR = 249
N_POP = 250
CANDIDATES = 1 + N_NEAR + N_POP

rng = np.random.default_rng(SEED)

# 5000 most popular TRAINING venues
top_pop_idx = (
    venue_meta
    .sort_values(["pop_count", "venue_id"], ascending=[False, True], kind="stable")
    .index[:5000]
    .to_numpy()
)

# Query more than 249 nearby places so that
# duplicates and the target itself can be skipped
query_coords = np.deg2rad(
    test_eligible[
        ["prev_lat", "prev_lon"]
    ].to_numpy()
)

_, nearest = tree.query(
    query_coords,
    k=N_NEAR + 50
)

candidate_sets = np.empty(
    (len(test_eligible), CANDIDATES),
    dtype=np.int32
)

for j, row in test_eligible.iterrows():

    # Actual visited POI
    target = venue_to_idx[row["venue_id"]]

    candidates = [target]
    seen = {target}

    # Add nearby POIs
    for x in nearest[j]:

        x = int(x)

        if x not in seen:
            candidates.append(x)
            seen.add(x)

        if len(candidates) >= 1 + N_NEAR:
            break

    # Add popularity-focused POIs
    while len(candidates) < CANDIDATES:

        sample_popular = rng.choice(
            top_pop_idx,
            size=min(400, len(top_pop_idx)),
            replace=False
        )

        for x in sample_popular:

            x = int(x)

            if x not in seen:
                candidates.append(x)
                seen.add(x)

            if len(candidates) >= CANDIDATES:
                break

    candidate_sets[j] = np.array(
        candidates,
        dtype=np.int32
    )

print("Candidate matrix shape:", candidate_sets.shape)
print("Candidates per event:", candidate_sets.shape[1])

Candidate matrix shape: (20223, 500)
Candidates per event: 500


In [129]:
# ==================================
# 14. BASE RECOMMENDER EVALUATION
# ==================================

base_rank = np.empty(
    len(test_eligible),
    dtype=int
)

for j, row in test_eligible.iterrows():

    candidates = candidate_sets[j]

    # Five interpretable features
    X = event_features(
        row,
        candidates
    )

    # Base recommender score
    scores = X @ weights

    # Sort candidates from highest to lowest score
    order = np.argsort(-scores, kind="stable")

    # Candidate position 0 is always the true visited POI
    rank = (
        np.where(order == 0)[0][0]
        + 1
    )

    base_rank[j] = rank


print("Evaluation finished.")
print("Test events:", len(base_rank))

Evaluation finished.
Test events: 20223


In [130]:
# ==================================
# 15. BASELINE METRICS
# ==================================

recall5 = np.mean(
    base_rank <= 5
)

ndcg5 = np.mean(
    np.where(
        base_rank <= 5,
        1 / np.log2(base_rank + 1),
        0
    )
)

mrr = np.mean(
    1 / base_rank
)

print("RankOnly results")
print("----------------")
print("Recall@5 :", recall5)
print("NDCG@5   :", ndcg5)
print("MRR      :", mrr)

RankOnly results
----------------
Recall@5 : 0.21495327102803738
NDCG@5   : 0.15652673088954747
MRR      : 0.15809918978621712


In [131]:
# ==========================================
# 16. APPLY CL-MEE TO THE REAL TEST SET
# ==========================================

M = 40
RHO = 0.80
ETA = 0.008

# B : (lambda, epsilon)
PARAMS = {
    3: (0.00, 0.00),
    2: (0.03, 0.04),
    1: (0.06, 0.08)
}

n_test = len(test_eligible)

# Results containers
cl_rank = {
    B: np.empty(n_test, dtype=int)
    for B in (1, 2, 3)
}

units = {
    B: np.empty(n_test, dtype=float)
    for B in (1, 2, 3)
}

fmodel = {
    B: np.empty(n_test, dtype=float)
    for B in (1, 2, 3)
}

pre_truncation_overbudget = {B: np.zeros(n_test, dtype=int) for B in (1, 2, 3)}

reranked = {
    B: np.zeros(n_test, dtype=int)
    for B in (1, 2, 3)
}

# Post-hoc minimal explanation
posthoc_units = np.empty(n_test, dtype=float)
posthoc_fmodel = np.empty(n_test, dtype=float)


for j, row in test_eligible.iterrows():

    candidates = candidate_sets[j]

    # Five interpretable features
    X = event_features(
        row,
        candidates
    )

    # Contribution of every feature
    contributions = (
        X * weights[None, :]
    )

    # Base score
    scores = contributions.sum(axis=1)

    # Base ranking
    order = np.argsort(-scores, kind="stable")

    # Only top-M candidates participate in CL-MEE
    top = order[:M]

    # Number of evidence units required for each top candidate
    K_values = np.empty(M, dtype=int)

    evidence_info = []

    for position, candidate_idx in enumerate(top):

        # Comparison alternative (paper convention):
        # base winner -> runner-up; any other top-M candidate -> base winner
        if candidate_idx != top[0]:
            alternative_idx = top[0]
        else:
            alternative_idx = top[1]

        # Positive pairwise evidence
        delta = np.maximum(
            0.0,
            contributions[candidate_idx]
            - contributions[alternative_idx]
        )

        total_evidence = float(delta.sum())

        sorted_features = np.argsort(-delta, kind="stable")

        if total_evidence <= 1e-12:

            # D=0 convention: K=m (five score features)
            K = contributions.shape[1]

        else:

            cumulative = (
                np.cumsum(
                    delta[sorted_features]
                )
                / total_evidence
            )

            K = (
                np.searchsorted(
                    cumulative,
                    RHO
                )
                + 1
            )

        K_values[position] = int(K)

        evidence_info.append(
            (
                sorted_features,
                delta,
                total_evidence
            )
        )

    # ==================================
    # Post-hoc minimal explanation
    # ==================================

    top_features, top_delta, top_total = evidence_info[0]

    posthoc_units[j] = K_values[0] if top_total > 1e-12 else 0

    if top_total > 1e-12:

        posthoc_fmodel[j] = (
            top_delta[
                top_features[:K_values[0]]
            ].sum()
            / top_total
        )

    else:
        posthoc_fmodel[j] = 0.0

    # ==================================
    # CL-MEE for B = 1, 2, 3
    # ==================================

    for B, (lam, epsilon) in PARAMS.items():

        best_score = float(
            scores[top[0]]
        )

        # Utility envelope
        eligible_positions = np.where(
            scores[top]
            >= best_score - epsilon
        )[0]

        # Explanation burden
        burden = np.maximum(
            0,
            K_values[eligible_positions] - B
        )

        # CL-MEE adjusted score
        adjusted_score = (
            scores[
                top[eligible_positions]
            ]
            - lam * burden
            - ETA
            * K_values[eligible_positions]
            / 5.0
        )

        # P3 is an explicit ranking identity control, even for exact score ties.
        chosen_position = (0 if B == 3 else int(
            eligible_positions[
                np.argmax(adjusted_score)
            ]
        ))

        chosen_candidate = top[
            chosen_position
        ]

        reranked[B][j] = int(
            chosen_position != 0
        )

        # Put selected candidate first,
        # keep remaining ranking order
        new_order = np.concatenate([
            [chosen_candidate],
            order[
                order != chosen_candidate
            ]
        ])

        # True POI is candidate index 0
        cl_rank[B][j] = (
            np.where(
                new_order == 0
            )[0][0]
            + 1
        )

        # Evidence actually displayed
        feature_order, delta, total = evidence_info[chosen_position]

        # K=m when D=0 is a penalty proxy, never a positive evidence count.
        displayed_units = (min(int(K_values[chosen_position]), B)
                           if total > 1e-12 else 0)
        units[B][j] = displayed_units
        pre_truncation_overbudget[B][j] = int(K_values[chosen_position] > B)

        if total > 1e-12:

            fmodel[B][j] = (
                delta[
                    feature_order[
                        :displayed_units
                    ]
                ].sum()
                / total
            )

        else:
            fmodel[B][j] = 0.0


print("CL-MEE evaluation finished.")
print("Test events:", n_test)

CL-MEE evaluation finished.
Test events: 20223


In [132]:
# ==========================================
# 17. CALCULATE FINAL CL-MEE RESULTS
# ==========================================

def recommendation_metrics(ranks):

    ranks = np.asarray(ranks)

    recall5 = np.mean(
        ranks <= 5
    )

    ndcg5 = np.mean(
        np.where(
            ranks <= 5,
            1 / np.log2(ranks + 1),
            0
        )
    )

    mrr = np.mean(
        1 / ranks
    )

    return recall5, ndcg5, mrr


results = []

# RankOnly
base_recall, base_ndcg, base_mrr = (
    recommendation_metrics(base_rank)
)

results.append({
    "Method": "RankOnly",
    "Recall@5": base_recall,
    "NDCG@5": base_ndcg,
    "MRR": base_mrr,
    "Units": np.nan,
    "F_model": np.nan,
    "Rerank_rate": 0.0,
    "NDCG_retained_%": 100.0
})

# Minimal post-hoc
results.append({
    "Method": "Minimal-posthoc",
    "Recall@5": base_recall,
    "NDCG@5": base_ndcg,
    "MRR": base_mrr,
    "Units": posthoc_units.mean(),
    "F_model": posthoc_fmodel.mean(),
    "Rerank_rate": 0.0,
    "NDCG_retained_%": 100.0
})

# CL-MEE
for B in (1, 2, 3):

    recall, ndcg, mrr_value = (
        recommendation_metrics(
            cl_rank[B]
        )
    )

    results.append({
        "Method": f"CL-MEE B={B}",
        "Recall@5": recall,
        "NDCG@5": ndcg,
        "MRR": mrr_value,
        "Units": units[B].mean(),
        "F_model": fmodel[B].mean(),
        "Rerank_rate": reranked[B].mean(),
        "NDCG_retained_%":
            100 * ndcg / base_ndcg
    })


results_df = pd.DataFrame(results)

display(results_df)
print("Overbudget before hard truncation:", {B: float(pre_truncation_overbudget[B].mean()) for B in (1,2,3)})
print("Display-budget violations:", {B: float(np.mean(units[B] > B)) for B in (1,2,3)})


,Method,Recall@5,NDCG@5,MRR,Units,F_model,Rerank_rate,NDCG_retained_%
0,RankOnly,0.214953,0.156527,0.158099,NaN,NaN,0.000000,100.000000
1,Minimal-posthoc,0.214953,0.156527,0.158099,1.303466,0.968066,0.000000,100.000000
2,CL-MEE B=1,0.214854,0.156520,0.158135,0.999951,0.912041,0.110073,99.995433
3,CL-MEE B=2,0.214953,0.156709,0.158346,1.270237,0.966221,0.012313,100.116594
4,CL-MEE B=3,0.214953,0.156527,0.158099,1.303120,0.968016,0.000000,100.000000


Overbudget before hard truncation: {1: 0.17351530435642584, 2: 0.013796172674677348, 3: 0.001483459427384661}
Display-budget violations: {1: 0.0, 2: 0.0, 3: 0.0}


In [133]:
# ==========================================
# 18. USER-LEVEL PAIRED STATISTICAL TESTS
# ==========================================

from scipy.stats import ttest_rel, wilcoxon

def ndcg5_contribution(ranks):
    ranks = np.asarray(ranks)

    return np.where(
        ranks <= 5,
        1 / np.log2(ranks + 1),
        0.0
    )


# Event-level NDCG contributions
base_ndcg_event = ndcg5_contribution(base_rank)

b1_ndcg_event = ndcg5_contribution(cl_rank[1])
b2_ndcg_event = ndcg5_contribution(cl_rank[2])
b3_ndcg_event = ndcg5_contribution(cl_rank[3])


# Build dataframe
stat_df = pd.DataFrame({
    "user_id": test_eligible["user_id"].to_numpy(),
    "RankOnly": base_ndcg_event,
    "B1": b1_ndcg_event,
    "B2": b2_ndcg_event,
    "B3": b3_ndcg_event
})


# Mean NDCG@5 for each user
user_ndcg = (
    stat_df
    .groupby("user_id")
    .mean()
)

print("Users in statistical test:", len(user_ndcg))

user_ndcg.head()

Users in statistical test: 1083


,RankOnly,B1,B2,B3
user_id,,,,
1,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000
4,0.233333,0.233333,0.233333,0.233333
5,0.298457,0.372271,0.298457,0.298457


In [134]:
# ==========================================
# 19. PAIRED T-TESTS
# ==========================================

for B in ["B1", "B2", "B3"]:

    test_result = ttest_rel(
        user_ndcg["RankOnly"],
        user_ndcg[B]
    )

    mean_difference = (
        user_ndcg[B]
        - user_ndcg["RankOnly"]
    ).mean()

    print(
        B,
        "| mean difference:",
        mean_difference,
        "| t:",
        test_result.statistic,
        "| p:",
        test_result.pvalue
    )

B1 | mean difference: -0.0002870257027784255 | t: 0.6447879035320029 | p: 0.5192013221787154
B2 | mean difference: 0.0001376403465662959 | t: -0.5894348486010592 | p: 0.55569265574205
B3 | mean difference: 0.0 | t: nan | p: nan


In [135]:
# ==========================================
# 20. WILCOXON PAIRED TEST
# ==========================================

for B in ["B1", "B2", "B3"]:

    differences = (
        user_ndcg[B]
        - user_ndcg["RankOnly"]
    )

    # Ignore users with exactly zero difference
    nonzero = differences[differences != 0]

    if len(nonzero) > 0:

        result = wilcoxon(nonzero)

        print(
            B,
            "| users with changed NDCG:",
            len(nonzero),
            "| W:",
            result.statistic,
            "| p:",
            result.pvalue
        )

    else:

        print(
            B,
            "| no NDCG differences"
        )

B1 | users with changed NDCG: 161 | W: 5810.5 | p: 0.23076517964589105
B2 | users with changed NDCG: 15 | W: 48.0 | p: 0.4954339467055269
B3 | no NDCG differences


## 4. Confidence intervals, policy labels, and final reproducibility checks

In [136]:

# 95% CIs for the user-level paired mean NDCG differences.
def paired_mean_ci(x, y, confidence=0.95):
    diff = np.asarray(y) - np.asarray(x)
    n = len(diff)
    mean = diff.mean()
    se = stats.sem(diff)
    crit = stats.t.ppf((1 + confidence) / 2, df=n - 1)
    return mean, (mean - crit * se, mean + crit * se)

for label in ["B1", "B2"]:
    mean_diff, ci = paired_mean_ci(user_ndcg["RankOnly"], user_ndcg[label])
    nonzero = (user_ndcg[label] - user_ndcg["RankOnly"])
    nonzero = nonzero[nonzero != 0]
    t_res = stats.ttest_rel(user_ndcg["RankOnly"], user_ndcg[label])
    w_res = stats.wilcoxon(nonzero)
    policy = {"B1":"P1", "B2":"P2"}[label]
    print(
        policy,
        "| mean diff =", mean_diff,
        "| 95% CI =", ci,
        "| t-test p =", t_res.pvalue,
        "| Wilcoxon p =", w_res.pvalue,
        "| non-zero users =", len(nonzero)
    )


P1 | mean diff = -0.0002870257027784255 | 95% CI = (np.float64(-0.0011604757625086866), np.float64(0.0005864243569518356)) | t-test p = 0.5192013221787154 | Wilcoxon p = 0.23076517964589105 | non-zero users = 161
P2 | mean diff = 0.0001376403465662959 | 95% CI = (np.float64(-0.0003205480852065193), np.float64(0.000595828778339111)) | t-test p = 0.55569265574205 | Wilcoxon p = 0.4954339467055269 | non-zero users = 15


In [137]:
# Reproducibility and structural assertions: no historic metric values are hardcoded.
assert len(test_eligible) == 20223
assert len(candidate_sets) == len(test_eligible)
assert candidate_sets.shape[1] == 500
assert all(np.all(units[B] <= B) for B in (1, 2, 3))
assert np.array_equal(cl_rank[3], base_rank), "P3 must be the ranking-identity control"
assert np.all((posthoc_units >= 0) & (posthoc_units <= 5))
assert all(np.all((fmodel[B] >= 0) & (fmodel[B] <= 1 + 1e-9)) for B in (1, 2, 3))
print("Structural protocol checks: PASS")
print("NDCG retention P1:", 100*recommendation_metrics(cl_rank[1])[1]/base_ndcg)


Structural protocol checks: PASS
NDCG retention P1: 99.99543328781776


In [138]:
# Export current-run values for paper Table 2, Table 3 and Table 4.
import json, sys
def jsfloat(x): return float(x)
paper_outputs = {
    "simulation_table2": sim_table2.to_dict(orient="records"),
    "sensitivity_detailed": sensitivity_detailed.to_dict(orient="records"),
    "simulation_summary": simulation_summary,
    "foursquare_table4": results_df.to_dict(orient="records"),
    "policy_stats": {},
    "pre_truncation_overbudget": {str(B):float(pre_truncation_overbudget[B].mean()) for B in (1,2,3)},
    "dataset_sha256": hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest(),
    "versions": {"python":platform.python_version(), "numpy":np.__version__, "pandas":pd.__version__, "scipy":scipy.__version__, "sklearn":sklearn.__version__}
}
for B in (1,2):
    label=f"B{B}"
    x=user_ndcg["RankOnly"]
    y=user_ndcg[label]
    dm,ci=paired_mean_ci(x,y)
    nonzero=(y-x)[(y-x)!=0]
    paper_outputs["policy_stats"][label]={"mean":float(dm),"ci":[float(v) for v in ci],"t_p":float(stats.ttest_rel(x,y).pvalue),"wilcoxon_p":float(stats.wilcoxon(nonzero).pvalue),"nonzero_users":int(len(nonzero))}
from pathlib import Path
def sanitize_json(v):
    if isinstance(v, dict): return {k: sanitize_json(z) for k, z in v.items()}
    if isinstance(v, list): return [sanitize_json(z) for z in v]
    if isinstance(v, float) and not np.isfinite(v): return None
    return v
Path("paper_results.json").write_text(json.dumps(sanitize_json(paper_outputs),indent=2,default=jsfloat,allow_nan=False),encoding="utf-8")
print("Saved paper_results.json")


Saved paper_results.json
